# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe
pandas: 2.3.3
Рабочая папка: c:\Users\User\Desktop\oocheba\2026\анализ данных 5 семестр\data-analysis-course-main\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied = home_outcomes.eq(3).sum()
home_denial_rate = home_denied / home_n
home_approved = home_outcomes.isin([1, 2]).sum()

print("Условие: action_taken == 3")
print("Числитель (отказы):", home_denied)
print("Знаменатель (все записи):", home_n)
print(f"Доля отказов: {100 * home_denial_rate:.2f}%")

assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1
print("Одобрено:", home_approved, "| баланс сходится, доля в диапазоне [0, 1]")

Условие: action_taken == 3
Числитель (отказы): 691
Знаменатель (все записи): 6830
Доля отказов: 10.12%
Одобрено: 6139 | баланс сходится, доля в диапазоне [0, 1]


## H2. Одобрение без принятия предложения

In [4]:
home_not_accepted = home_outcomes.eq(2).sum()
home_not_accepted_rate = home_not_accepted / home_n
print("Условие: action_taken == 2")
print("Числитель:", home_not_accepted, "| знаменатель:", home_n)
print(f"Доля «одобрено, предложение не принято»: {100 * home_not_accepted_rate:.2f}%")

home_issued = home_outcomes.eq(1).sum()
assert home_approved - home_issued == home_not_accepted
print("Проверка по количествам:", home_approved, "-", home_issued, "=", home_not_accepted)

home_approval_rate = home_approved / home_n
home_origination_rate = home_issued / home_n
diff_pp = 100 * (home_approval_rate - home_origination_rate)
print(f"Одобрения {100 * home_approval_rate:.2f}% - выдачи {100 * home_origination_rate:.2f}% = {diff_pp:.2f} п.п.")
print(f"Доля исхода 2: {100 * home_not_accepted_rate:.2f}% (совпадает с разностью)")

Условие: action_taken == 2
Числитель: 183 | знаменатель: 6830
Доля «одобрено, предложение не принято»: 2.68%
Проверка по количествам: 6139 - 5956 = 183
Одобрения 89.88% - выдачи 87.20% = 2.68 п.п.
Доля исхода 2: 2.68% (совпадает с разностью)


## H3. Объяснение инструментов

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

In [6]:
raw_issued = raw_outcomes.eq(1).sum()
raw_issued_rate = raw_issued / raw_n
home_issued_rate = home_issued / home_n

print("raw:           ", raw_issued, "/", raw_n, f"= {100 * raw_issued_rate:.2f}%")
print("home_decisions:", home_issued, "/", home_n, f"= {100 * home_issued_rate:.2f}%")
assert raw_issued == home_issued
print("Количество выдач в двух файлах совпадает; различается только знаменатель")

raw:            5956 / 8575 = 69.46%
home_decisions: 5956 / 6830 = 87.20%
Количество выдач в двух файлах совпадает; различается только знаменатель


## Текстовые коды для H+2

In [7]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом

In [8]:
as_number = text_example["action_taken"].eq(1)
as_text = text_example["action_taken"].eq("1")
display(as_number)
display(as_text)
print("eq(1):  ", as_number.sum(), "True")
print('eq("1"):', as_text.sum(), "True")

0    False
1    False
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean

0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean

eq(1):   0 True
eq("1"): 2 True
